
<table align="center" style="margin: 0 auto; border: 0; width: auto;">
<tr><td align="center" style="border: 0; padding: 0; text-align: center;">
<img src="https://logoyar.com/content/wp-content/uploads/2021/04/sharif-university-logo.png" width="155" alt="Sharif University of Technology logo" style="background: white; padding: 8px; border-radius: 50%;" />
</td></tr></table>

<h2 align="center">SHARIF UNIVERSITY OF TECHNOLOGY</h2>

<h3 align="center">Department of Electrical Engineering</h3>


<h1 align="center">Deep Generative Models</h1>

<h2 align="center">Practical Assignment 2 | Notebook 2 | Normalizing Flows</h2>






<table align="center" style="margin: 15px auto 25px auto; width: 65%; border-collapse: collapse; text-align: center;">
  <tr>
    <th width="50%" style="text-align: center; padding: 12px; border-bottom: 1px solid;">
      Student Name
    </th>
    <th width="50%" style="text-align: center; padding: 12px; border-bottom: 1px solid;">
      Student ID
    </th>
  </tr>
  <tr>
    <td style="text-align: center; padding: 20px 12px;">
      ______________________
    </td>
    <td style="text-align: center; padding: 20px 12px;">
      ______________________
    </td>
  </tr>
</table>


---

### **Assignment Overview & Instructions:**



**Aim.** In this assignment you will implement several flow-based components and study three applications:
1. anomaly localization on healthy-only training data,
2. conditional colorization from grayscale luminance,
3. global color-palette transfer in CIE Lab space.

| Part | Topic | Points |
|---|---|---:|
| I | Anomaly Detection | 60 |
| II | Conditional Colorization | 30 |
| III | Color Transfer | 10 |
|  | **Total** | **100** |

**General instructions**
- Write your code only inside the marked `TODO` regions.
- Keep all random seeds and configurations unchanged unless a section explicitly asks otherwise.
- Run the correctness checks before running the full experiments.
- For a quick structure test, you may temporarily set `FAST_DEV_RUN=True`. Use `False` for the final submitted notebook.
- Submit one executed notebook with your implementations, figures, numerical outputs, and written answers.

**Environment requirements**
- Python 3.10+
- PyTorch, torchvision, numpy, matplotlib, pandas, scikit-learn, scikit-image, scipy
- Kaggle GPU is recommended for the full experiments, but the notebook also supports CPU execution.

This homework follows the experiments that we already validated during course preparation.
You will implement the main flow components instead of only running a demonstration.
The notebook contains three applications:

- **Part I:** anomaly detection on brick texture and real MVTec AD images,
- **Part II:** conditional colorization on **STL-10**,
- **Part III:** global color transfer using a 2D masked autoregressive flow.

The anomaly-detection parts show how likelihood on healthy data can be used for localization.
The colorization part shows how a conditional flow can model multiple plausible chrominance values for one grayscale image.
The final part is intentionally easier: it changes the global color palette, but it does not understand object semantics.

### Notes on validation

- `FAST_DEV_RUN=True` is intended only for debugging the full pipeline.
- For the final report, set `FAST_DEV_RUN=False` and run all required sections.


In [ ]:

from pathlib import Path
import os, sys, math, time, random, json, subprocess, gc
import numpy as np
import pandas as pd
from IPython.display import display
import matplotlib.pyplot as plt
from PIL import Image

import torch
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets, transforms, models
from torchvision.models import ResNet18_Weights

from scipy.ndimage import gaussian_filter
from sklearn.metrics import roc_auc_score, average_precision_score
from skimage import data as skdata, color as skcolor
from skimage.draw import disk, line
from skimage.transform import resize as skresize

SEED = 314
FAST_DEV_RUN = False        # True: quick debugging only, not the final report configuration.
NUM_WORKERS = 0 if os.name == 'nt' else 2

# ---------- Part I: Anomaly Detection ----------
BRICK_PATCH_SIZE = 96
BRICK_TRAIN_PATCHES = 320 if not FAST_DEV_RUN else 64
BRICK_FLOW_STEPS = 450 if not FAST_DEV_RUN else 30
BRICK_BATCH_SIZE = 16
MVTEC_CATEGORIES = ['tile', 'hazelnut', 'wood', 'bottle']
MVTEC_IMAGE_SIDE = 192 if not FAST_DEV_RUN else 128
MVTEC_FLOW_STEPS = 500 if not FAST_DEV_RUN else 35
MVTEC_BATCH_SIZE = 12 if not FAST_DEV_RUN else 8
MVTEC_PROJECTED_DIM = 32
MVTEC_FUSION_WEIGHTS = (0.65, 0.35)
MVTEC_FEATURE_SAMPLES = 15000 if not FAST_DEV_RUN else 1500

# ---------- Part II: Colorization ----------
IMAGE_SIZE = 96
CHROMA_SIZE = 48
STL_BATCH_SIZE = 16 if not FAST_DEV_RUN else 8
COND_WIDTH = 48
FLOW_STEPS = 10
FLOW_HIDDEN = 64
COLORIZATION_TRAIN_STEPS = 3500 if not FAST_DEV_RUN else 25
VALIDATE_EVERY = 500 if not FAST_DEV_RUN else 10
SAVE_EVERY = 1000 if not FAST_DEV_RUN else 25
LOG_EVERY = 100 if not FAST_DEV_RUN else 5
NUM_VIS_IMAGES = 6
NUM_SAMPLES = 4
TEMPERATURES = (0.65, 0.9, 1.1)
LR = 2e-4
GRAD_CLIP = 5.0

# ---------- Part III: Color Transfer ----------
MAF_STEPS = 1100 if not FAST_DEV_RUN else 120

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
if not torch.cuda.is_available():
    torch.set_num_threads(min(4, os.cpu_count() or 1))

def detect_device():
    if torch.cuda.is_available():
        return torch.device('cuda')
    if hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return torch.device('mps')
    return torch.device('cpu')

DEVICE = detect_device()
ROOT = Path('/kaggle/working/dgm_flow_assignment') if Path('/kaggle/working').exists() else Path('./dgm_flow_assignment')
DATA_ROOT = ROOT / 'datasets'
FIG_ROOT = ROOT / 'figures'
CKPT_ROOT = ROOT / 'checkpoints'
for p in [ROOT, DATA_ROOT, FIG_ROOT, CKPT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

print('Device:', DEVICE)
print('Output directory:', ROOT.resolve())

def savefig(name, dpi=160, show=True):
    plt.tight_layout()
    plt.savefig(FIG_ROOT / name, dpi=dpi, bbox_inches='tight')
    if show:
        plt.show()
    else:
        plt.close()

def release_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


# **Part I — Anomaly Detection (60 points)**

## I.1 — Synthetic Defects (25 points)

We start with a controlled experiment on a real brick texture. Training images are **healthy only**. At test time we insert localized synthetic defects into unseen brick patches.
This part is useful because the correct localization target is unambiguous.

We first extract a small fixed feature tensor from each image using average pooling and Sobel filters. A small spatial flow is then trained on healthy feature maps.
The local anomaly score is the negative log-likelihood of each spatial feature vector.

For one spatial location with latent vector $z \in \mathbb{R}^C$ and accumulated log-determinant $\log |\det J|$, the pointwise negative log-likelihood is
\[
\mathcal{S}(x) = \sum_{c=1}^{C} \frac{1}{2}\left(z_c^2 + \log(2\pi)\right) - \log |\det J|.
\]

In this part you will implement the forward pass of an affine coupling layer and the score map of the full spatial flow.


In [ ]:

# Data construction for Part I.
texture = (skdata.brick().astype('float32') / 255.0)
assert texture.ndim == 2
_rng_global = np.random.default_rng(SEED)

def sample_patch(rng, sz=BRICK_PATCH_SIZE):
    i = int(rng.integers(0, texture.shape[0] - sz))
    j = int(rng.integers(0, texture.shape[1] - sz))
    x = texture[i:i+sz, j:j+sz].copy()
    if rng.random() < 0.5:
        x = np.fliplr(x).copy()
    if rng.random() < 0.5:
        x = np.flipud(x).copy()
    return x.astype('float32')

class HealthyTextures(Dataset):
    def __init__(self, n=BRICK_TRAIN_PATCHES, seed=SEED):
        self.n = n
        self.seed = seed
    def __len__(self):
        return self.n
    def __getitem__(self, idx):
        rng = np.random.default_rng(self.seed + 17*idx)
        x = sample_patch(rng)
        return torch.from_numpy(x[None].copy())

def inject_defect(im, seed=0):
    rng = np.random.default_rng(seed)
    corrupted = im.copy()
    mask = np.zeros(im.shape, dtype='float32')

    rr, cc = disk((int(rng.integers(28, 68)), int(rng.integers(28, 68))),
                  int(rng.integers(8, 14)), shape=im.shape)
    mask[rr, cc] = 1.0
    corrupted[rr, cc] = 0.95

    rr, cc = line(20, 76, 77, 49)
    rr = np.clip(rr, 0, im.shape[0]-1)
    cc = np.clip(cc, 0, im.shape[1]-1)
    for offset in [-1, 0, 1]:
        c = np.clip(cc + offset, 0, im.shape[1]-1)
        mask[rr, c] = 1.0
        corrupted[rr, c] = 0.02
    return corrupted.astype('float32'), mask

sobel_x = torch.tensor([[-1,0,1],[-2,0,2],[-1,0,1]], dtype=torch.float32).view(1,1,3,3) / 8.0
sobel_y = sobel_x.transpose(-1, -2).contiguous()

def fixed_features(img):
    img = F.avg_pool2d(img, 3, stride=3)
    sm = F.avg_pool2d(img, 5, stride=1, padding=2)
    dx = F.conv2d(img, sobel_x.to(img.device), padding=1)
    dy = F.conv2d(img, sobel_y.to(img.device), padding=1)
    return torch.cat([img, dx, dy, img-sm], dim=1)

brick_loader = DataLoader(HealthyTextures(), batch_size=BRICK_BATCH_SIZE, shuffle=True, num_workers=0)
with torch.no_grad():
    train_feature_stack = torch.cat([fixed_features(v) for v in brick_loader], dim=0)
    brick_mean = train_feature_stack.mean((0,2,3), keepdim=True)
    brick_std = train_feature_stack.std((0,2,3), keepdim=True).clamp_min(0.015)
    train_features = ((train_feature_stack - brick_mean) / brick_std).clamp(-7, 7)

print('Healthy brick feature stack:', tuple(train_feature_stack.shape))
preview = next(iter(brick_loader))
fig, ax = plt.subplots(1, 4, figsize=(10, 2.8))
for i in range(4):
    ax[i].imshow(preview[i,0], cmap='gray', vmin=0, vmax=1)
    ax[i].axis('off')
    ax[i].set_title(f'Patch {i+1}')
savefig('part1_healthy_preview.png')


In [ ]:

class SpatialCoupling(nn.Module):
    def __init__(self, channels=4, hidden=32, flip=False):
        super().__init__()
        self.flip = flip
        self.net = nn.Sequential(
            nn.Conv2d(channels//2, hidden, 3, padding=1), nn.ReLU(),
            nn.Conv2d(hidden, hidden, 1), nn.ReLU(),
            nn.Conv2d(hidden, channels, 3, padding=1),
        )
        nn.init.zeros_(self.net[-1].weight)
        nn.init.zeros_(self.net[-1].bias)

    def forward(self, x):
        """
        Input:
            x: tensor of shape [B, C, H, W] with even C.
        Output:
            z: transformed tensor with the same shape.
            logdet_map: tensor of shape [B, H, W].
        """
        # TODO I.1 (10 points): implement the affine coupling forward pass.
        # Requirements:
        # 1. If self.flip is True, reverse the channel order before splitting.
        # 2. Split x into xa and xb along the channel dimension.
        # 3. Use self.net(xa) and split it into translation t and scale pre-activation s.
        # 4. Bound the log-scale with 1.2 * tanh(s / 1.2).
        # 5. Transform xb into latent zb = (xb - t) * exp(-s).
        # 6. Concatenate xa and zb, undo the channel flip if needed, and return
        #    the output together with logdet_map = -sum_c s.
        raise NotImplementedError('TODO I.1')

class SpatialFeatureFlow(nn.Module):
    def __init__(self, channels=4, steps=6, hidden=32):
        super().__init__()
        self.layers = nn.ModuleList([
            SpatialCoupling(channels=channels, hidden=hidden, flip=(i % 2 == 1))
            for i in range(steps)
        ])

    def score_map(self, x):
        """
        Input:
            x: [B, C, H, W]
        Output:
            score: [B, H, W] pointwise negative log-likelihood.
        """
        # TODO I.2 (8 points): compute the pointwise negative log-likelihood map.
        # Requirements:
        # 1. Apply all coupling layers in order.
        # 2. Accumulate the per-location log-determinant terms.
        # 3. Use a standard Gaussian base density independently at each location.
        # 4. Return the negative log-likelihood map with shape [B, H, W].
        raise NotImplementedError('TODO I.2')

# Correctness checks for Part I.
_test_flow = SpatialFeatureFlow(channels=4, steps=3, hidden=16)
_test_x = torch.randn(5, 4, 11, 13)
_test_score = _test_flow.score_map(_test_x)
assert _test_score.shape == (5, 11, 13), 'score_map must return [B, H, W]'
assert torch.isfinite(_test_score).all(), 'score_map returned non-finite values'
print('Part I shape and finiteness checks passed.')


In [ ]:

brick_flow = SpatialFeatureFlow(channels=train_features.shape[1], steps=6, hidden=32).to(DEVICE)
optimizer = torch.optim.Adam(brick_flow.parameters(), lr=5e-4)
brick_hist = []
brick_feats = train_features.to(DEVICE)
for step in range(1, BRICK_FLOW_STEPS + 1):
    ids = torch.randint(0, len(brick_feats), (BRICK_BATCH_SIZE,), device=DEVICE)
    batch = brick_feats[ids] + 0.018 * torch.randn_like(brick_feats[ids])
    loss = brick_flow.score_map(batch).mean()
    if not torch.isfinite(loss):
        raise RuntimeError(f'Brick flow diverged at step {step}')
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    nn.utils.clip_grad_norm_(brick_flow.parameters(), 5.0)
    optimizer.step()
    brick_hist.append(float(loss.detach().cpu()))
    if step == 1 or step % max(100, BRICK_FLOW_STEPS) == 0 or step == BRICK_FLOW_STEPS:
        print(f'brick step {step}/{BRICK_FLOW_STEPS} | NLL {np.mean(brick_hist[-25:]):.3f}')

plt.figure(figsize=(6,3))
plt.plot(brick_hist)
plt.xlabel('Training step')
plt.ylabel('Mean pointwise NLL')
plt.title('Part I training curve')
savefig('part1_training_curve.png')

def evaluate_brick_maps(model, images_np):
    x = torch.from_numpy(images_np[:, None]).to(DEVICE)
    with torch.no_grad():
        f = ((fixed_features(x) - brick_mean.to(DEVICE)) / brick_std.to(DEVICE)).clamp(-7, 7)
        score = model.score_map(f).unsqueeze(1)
        score = F.interpolate(score, size=x.shape[-2:], mode='bilinear', align_corners=False)
    return score.squeeze(1).cpu().numpy()

rng = np.random.default_rng(111)
originals = [sample_patch(rng) for _ in range(8)]
corrupted_and_masks = [inject_defect(x, seed=SEED+i) for i, x in enumerate(originals)]
corrupted = np.stack([a for a, m in corrupted_and_masks])
masks = np.stack([m for a, m in corrupted_and_masks])

normal_scores = evaluate_brick_maps(brick_flow, np.stack(originals))
abnormal_scores = evaluate_brick_maps(brick_flow, corrupted)
brick_pixel_auroc = roc_auc_score(masks.reshape(-1), abnormal_scores.reshape(-1))
print('Brick pixel AUROC:', round(float(brick_pixel_auroc), 4))

fig, ax = plt.subplots(2, 4, figsize=(12, 6), squeeze=False)
for row in range(2):
    ax[row, 0].imshow(originals[row], cmap='gray', vmin=0, vmax=1)
    ax[row, 0].set_title('Healthy patch')
    ax[row, 1].imshow(corrupted[row], cmap='gray', vmin=0, vmax=1)
    ax[row, 1].set_title('Defective patch')
    ax[row, 2].imshow(masks[row], cmap='gray', vmin=0, vmax=1)
    ax[row, 2].set_title('True mask')
    ax[row, 3].imshow(corrupted[row], cmap='gray', vmin=0, vmax=1)
    ax[row, 3].imshow(abnormal_scores[row], cmap='inferno', alpha=0.55)
    ax[row, 3].set_title('Flow anomaly map')
    for j in range(4):
        ax[row, j].axis('off')
savefig('part1_brick_results.png')


**Question I.3 (7 points).** Compare the anomaly maps with the true defect masks.
Why does the model localize the inserted defects reasonably well, and what limitation still remains in this controlled experiment?

**Answer:** _Write your answer here._


## I.2 — Real Defects (35 points)

We now move to real industrial anomalies. The setup remains **healthy-only training**.
For each category, only `train/good` images are used for fitting the model. Test images may be healthy or defective.

We reuse the same spatial flow idea, but now on feature maps extracted from a frozen pretrained ResNet-18.
We use two spatial resolutions:
- a **shallow** feature map, which keeps more local detail,
- a **deep** feature map, which is more semantic but spatially coarser.

Before fitting the flow, each feature vector is projected to a lower dimension with PCA learned from healthy training features.
You will implement PCA fitting, evaluation metrics, and score calibration.


In [ ]:

def is_mvtec_category(p):
    p = Path(p)
    return (p/'train'/'good').is_dir() and (p/'test').is_dir()

def find_mvtec_category(category):
    roots = [DATA_ROOT/'mvtec_ad', Path('/kaggle/input'), Path('/kaggle/working'), Path.cwd()/'datasets']
    for root in roots:
        if not root.exists():
            continue
        candidates = [root/category, root/'mvtec_ad'/category, root/'flow_anomaly_v5'/'mvtec_ad'/category]
        for path in candidates:
            if is_mvtec_category(path):
                return path
        if root == Path('/kaggle/input'):
            for path in root.glob(f'**/{category}/train/good'):
                candidate = path.parent.parent
                if is_mvtec_category(candidate):
                    return candidate
    return None

def ensure_mvtec_category(category):
    existing = find_mvtec_category(category)
    if existing is not None:
        print('Using cached MVTec category:', existing)
        return existing
    try:
        from huggingface_hub import snapshot_download
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'huggingface_hub'], check=True)
        from huggingface_hub import snapshot_download
    target_root = DATA_ROOT / 'mvtec_ad'
    target_root.mkdir(parents=True, exist_ok=True)
    snapshot_download(
        repo_id='foersben/mvtec-ad', repo_type='dataset', local_dir=str(target_root),
        allow_patterns=[f'{category}/train/**', f'{category}/test/**', f'{category}/ground_truth/**'],
        max_workers=4,
    )
    found = find_mvtec_category(category)
    if found is None:
        raise FileNotFoundError(f'MVTec category {category} could not be located after download.')
    return found

class MVTecImages(Dataset):
    def __init__(self, category_folder, split='train', side=MVTEC_IMAGE_SIDE):
        self.side = side
        self.items = []
        category_folder = Path(category_folder)
        assert split in ('train', 'test')
        for sub in sorted((category_folder / split).iterdir()):
            if not sub.is_dir():
                continue
            if split == 'train' and sub.name != 'good':
                continue
            for img_path in sorted(sub.glob('*.png')):
                is_bad = sub.name != 'good'
                mask_path = category_folder/'ground_truth'/sub.name/(img_path.stem + '_mask.png')
                if is_bad and not mask_path.is_file():
                    raise FileNotFoundError(f'Missing mask: {mask_path}')
                self.items.append((img_path, mask_path if is_bad else None, is_bad))
        if not self.items:
            raise ValueError(f'No {split} images found in {category_folder}')

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        img_path, mask_path, is_bad = self.items[index]
        x = transforms.functional.to_tensor(
            Image.open(img_path).convert('RGB').resize((self.side, self.side), Image.Resampling.BILINEAR)
        )
        if mask_path is None:
            mask = np.zeros((self.side, self.side), dtype=bool)
        else:
            mask = np.asarray(
                Image.open(mask_path).convert('L').resize((self.side, self.side), Image.Resampling.NEAREST)
            ) > 127
        return x, torch.from_numpy(mask.copy()).float(), int(is_bad), str(img_path)

# Frozen ResNet-18 features, using the same layers as the validated pilot.
try:
    resnet = models.resnet18(weights=ResNet18_Weights.IMAGENET1K_V1).to(DEVICE).eval()
except Exception as exc:
    raise RuntimeError('ResNet18 weights were not available. Check network or cached weights.') from exc
for p in resnet.parameters():
    p.requires_grad_(False)

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)

@torch.no_grad()
def resnet_features(images):
    x = (images - IMAGENET_MEAN) / IMAGENET_STD
    x = resnet.maxpool(resnet.relu(resnet.bn1(resnet.conv1(x))))
    shallow = resnet.layer1(x)         # ~48x48 for 192x192 input
    deep = resnet.layer2(shallow)     # ~24x24 for 192x192 input
    return {'shallow': shallow, 'deep': deep}

print('MVTec ResNet scales:', {k: tuple(v.shape) for k,v in resnet_features(torch.rand(1,3,MVTEC_IMAGE_SIDE,MVTEC_IMAGE_SIDE,device=DEVICE)).items()})


In [ ]:

class HealthyPCA:
    def __init__(self, out_dim=MVTEC_PROJECTED_DIM):
        self.out_dim = out_dim
        self.mean = None
        self.components = None
        self.scale = None

    def fit(self, x):
        """
        x: [N, D] healthy feature vectors.
        Fit PCA on centered data and store:
            self.mean: [1, D]
            self.components: [D, out_dim]
            self.scale: [1, out_dim]
        Use torch.pca_lowrank.
        """
        # TODO I.4 (10 points): fit PCA on healthy features.
        # Requirements:
        # 1. Compute and store the mean.
        # 2. Center the data and run torch.pca_lowrank.
        # 3. Keep the first out_dim principal directions.
        # 4. Store a per-dimension scale from the projected healthy data.
        raise NotImplementedError('TODO I.4')

    def transform(self, x):
        """Project x to the PCA subspace and normalize each projected dimension."""
        # TODO I.4 continuation (4 of the 10 points): implement the projection.
        raise NotImplementedError('TODO I.4 transform')

# PCA sanity checks.
_x = torch.randn(300, 64)
_pca = HealthyPCA(out_dim=16)
_pca.fit(_x)
_z = _pca.transform(_x)
assert _z.shape == (300, 16)
assert torch.isfinite(_z).all()
print('Part I PCA checks passed.')


In [ ]:

def collect_projected_feature_maps(loader, pca_modules):
    outputs = {k: [] for k in pca_modules}
    masks_all, labels_all, paths_all, images_all = [], [], [], []
    with torch.no_grad():
        for images, masks, labels, paths in loader:
            images = images.to(DEVICE)
            feats = resnet_features(images)
            for key, fmap in feats.items():
                b, c, h, w = fmap.shape
                flat = fmap.permute(0, 2, 3, 1).reshape(-1, c).cpu()
                z = pca_modules[key].transform(flat)
                z = z.reshape(b, h, w, -1).permute(0, 3, 1, 2).contiguous()
                outputs[key].append(z)
            masks_all.extend(masks.numpy())
            labels_all.extend(labels.numpy())
            paths_all.extend(list(paths))
            images_all.extend(images.cpu().permute(0,2,3,1).numpy())
    outputs = {k: torch.cat(v, dim=0).to(DEVICE) for k, v in outputs.items()}
    return outputs, np.asarray(masks_all), np.asarray(labels_all).astype(int), paths_all, np.asarray(images_all)

def fit_pca_from_training_set(train_loader):
    shallow_vectors, deep_vectors = [], []
    with torch.no_grad():
        for images, _, _, _ in train_loader:
            feats = resnet_features(images.to(DEVICE))
            shallow_vectors.append(feats['shallow'].permute(0,2,3,1).reshape(-1, feats['shallow'].shape[1]).cpu())
            deep_vectors.append(feats['deep'].permute(0,2,3,1).reshape(-1, feats['deep'].shape[1]).cpu())
    shallow_vectors = torch.cat(shallow_vectors, dim=0)
    deep_vectors = torch.cat(deep_vectors, dim=0)
    if len(shallow_vectors) > MVTEC_FEATURE_SAMPLES:
        idx = torch.randperm(len(shallow_vectors))[:MVTEC_FEATURE_SAMPLES]
        shallow_vectors = shallow_vectors[idx]
    if len(deep_vectors) > MVTEC_FEATURE_SAMPLES:
        idx = torch.randperm(len(deep_vectors))[:MVTEC_FEATURE_SAMPLES]
        deep_vectors = deep_vectors[idx]
    pcas = {'shallow': HealthyPCA(MVTEC_PROJECTED_DIM), 'deep': HealthyPCA(MVTEC_PROJECTED_DIM)}
    pcas['shallow'].fit(shallow_vectors)
    pcas['deep'].fit(deep_vectors)
    return pcas

def fit_flow_on_feature_maps(projected_train_maps, key):
    model = SpatialFeatureFlow(channels=MVTEC_PROJECTED_DIM, steps=5, hidden=80 if not FAST_DEV_RUN else 48).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=5e-4)
    x = projected_train_maps[key]
    history = []
    for step in range(1, MVTEC_FLOW_STEPS + 1):
        ids = torch.randint(0, len(x), (MVTEC_BATCH_SIZE,), device=DEVICE)
        batch = x[ids] + 0.01 * torch.randn_like(x[ids])
        loss = model.score_map(batch).mean()
        if not torch.isfinite(loss):
            raise RuntimeError(f'Non-finite MVTec loss for {key} at step {step}')
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return model.eval(), history

def smooth_map_array(x, sigma=1.3):
    return np.stack([gaussian_filter(m, sigma=sigma) for m in x], axis=0)

def pixel_and_image_metrics(masks, labels, score_map):
    """
    masks: [N, H, W] binary masks.
    labels: [N] image labels, 1 for defective.
    score_map: [N, H, W] anomaly score.

    Return a dictionary with:
        pixel AUROC,
        pixel AUPRC,
        image AUROC.
    Use the max score over spatial positions as the image-level score.
    """
    # TODO I.5 (10 points): compute the requested metrics.
    raise NotImplementedError('TODO I.5')

def estimate_healthy_score_stats(model, healthy_maps, batch_size=8):
    """Estimate mean/std of flow anomaly scores on HEALTHY TRAINING maps only.

    Input: model and healthy_maps [N, C, H, W].
    Return: (mean, std), two scalar values. No test images or masks may be used.
    """
    # TODO I.6 (4 points): estimate the training-score mean and standard deviation.
    # Use model.score_map() under torch.no_grad() in small batches.
    # Return finite scalars; clamp standard deviation to at least 1e-5.
    raise NotImplementedError('TODO I.6')

# Check with a simple controlled model whose output equals one input channel.
class _ControlledScore(nn.Module):
    def score_map(self, x):
        return x[:, 0]
_controlled = torch.arange(20, dtype=torch.float32).view(5,1,2,2)
_cal_mu, _cal_sd = estimate_healthy_score_stats(_ControlledScore(), _controlled, batch_size=2)
assert abs(float(_cal_mu) - 9.5) < 1e-5
assert abs(float(_cal_sd) - float(_controlled[:,0].std(unbiased=False))) < 1e-5
print('Healthy-score calibration check passed.')

def choose_visual_examples(category, paths, masks, shallow_scores, fused_scores, top_k=2):
    """For illustration ONLY; results in the metrics table use every test image.

    Tile uses the first three real defective images from the successful v3 pilot.
    Other categories show the strongest examples under a pixel-localization metric.
    """
    positive = np.where(masks.reshape(len(masks), -1).sum(axis=1) > 0)[0]
    if len(positive) == 0:
        return []
    if category == 'tile':
        return list(positive[:3])
    quality = []
    for idx in positive:
        y = masks[idx].reshape(-1) > 0.5
        if y.sum() < 3 or (~y).sum() < 3:
            continue
        ap_shallow = average_precision_score(y, shallow_scores[idx].reshape(-1))
        ap_fused = average_precision_score(y, fused_scores[idx].reshape(-1))
        quality.append((max(ap_shallow, ap_fused), int(idx)))
    quality.sort(key=lambda z: z[0], reverse=True)
    return [idx for _, idx in quality[:top_k]]


In [ ]:
mvtec_rows = []
for category in MVTEC_CATEGORIES:
    print('\n' + '='*18, category, '='*18)
    category_root = ensure_mvtec_category(category)
    train_set = MVTecImages(category_root, split='train', side=MVTEC_IMAGE_SIDE)
    test_set = MVTecImages(category_root, split='test', side=MVTEC_IMAGE_SIDE)
    train_loader_mv = DataLoader(train_set, batch_size=MVTEC_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    test_loader_mv = DataLoader(test_set, batch_size=MVTEC_BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    pcas = fit_pca_from_training_set(train_loader_mv)
    projected_train, _, _, _, _ = collect_projected_feature_maps(train_loader_mv, pcas)
    projected_test, masks_np, labels_np, paths_np, images_np = collect_projected_feature_maps(test_loader_mv, pcas)

    flows, flow_cal, base_cal = {}, {}, {}
    score_maps, baseline_maps = {}, {}
    for key in ['shallow', 'deep']:
        flows[key], _ = fit_flow_on_feature_maps(projected_train, key)
        flow_cal[key] = estimate_healthy_score_stats(flows[key], projected_train[key])
        with torch.no_grad():
            feature_energy = projected_train[key].square().mean(dim=1).flatten()
            base_cal[key] = (float(feature_energy.mean()), max(float(feature_energy.std(unbiased=False)), 1e-5))
            raw_flow = flows[key].score_map(projected_test[key])[:,None]
            raw_base = projected_test[key].square().mean(dim=1, keepdim=True)
            raw_flow = F.interpolate(raw_flow, size=(MVTEC_IMAGE_SIDE, MVTEC_IMAGE_SIDE), mode='bilinear', align_corners=False)[:,0]
            raw_base = F.interpolate(raw_base, size=(MVTEC_IMAGE_SIDE, MVTEC_IMAGE_SIDE), mode='bilinear', align_corners=False)[:,0]
            # Standardization prevents the coarse map from overpowering the shallow map.
            score_maps[key] = smooth_map_array(((raw_flow-flow_cal[key][0])/flow_cal[key][1]).cpu().numpy(), sigma=1.3)
            baseline_maps[key] = smooth_map_array(((raw_base-base_cal[key][0])/base_cal[key][1]).cpu().numpy(), sigma=1.3)

    fused = MVTEC_FUSION_WEIGHTS[0]*score_maps['shallow'] + MVTEC_FUSION_WEIGHTS[1]*score_maps['deep']
    fused_baseline = MVTEC_FUSION_WEIGHTS[0]*baseline_maps['shallow'] + MVTEC_FUSION_WEIGHTS[1]*baseline_maps['deep']
    metrics_fused = pixel_and_image_metrics(masks_np, labels_np, fused)
    metrics_base = pixel_and_image_metrics(masks_np, labels_np, fused_baseline)
    row = {
        'category': category,
        'train_images': len(train_set),
        'test_images': len(test_set),
        'shallow pixel AUROC': pixel_and_image_metrics(masks_np, labels_np, score_maps['shallow'])['pixel AUROC'],
        'deep pixel AUROC': pixel_and_image_metrics(masks_np, labels_np, score_maps['deep'])['pixel AUROC'],
        'fused pixel AUROC': metrics_fused['pixel AUROC'],
        'fused pixel AUPRC': metrics_fused['pixel AUPRC'],
        'baseline pixel AUROC': metrics_base['pixel AUROC'],
        'image AUROC': metrics_fused['image AUROC'],
    }
    mvtec_rows.append(row)
    np.savez_compressed(ROOT/f'mvtec_{category}_test_scores.npz',
        mask=masks_np, labels=labels_np, paths=np.asarray(paths_np),
        shallow=score_maps['shallow'], deep=score_maps['deep'], fused=fused, baseline=fused_baseline)

    chosen = choose_visual_examples(category, paths_np, masks_np, score_maps['shallow'], fused, top_k=2)
    if chosen:
        fig, ax = plt.subplots(len(chosen), 4, figsize=(12, 3.1*len(chosen)), squeeze=False)
        vmin = float(np.quantile(fused, 0.12))
        vmax = float(np.quantile(fused, 0.995))
        selection_rows=[]
        for r, idx in enumerate(chosen):
            ax[r,0].imshow(images_np[idx]); ax[r,0].set_title('Defective image')
            ax[r,1].imshow(masks_np[idx], cmap='gray', vmin=0, vmax=1); ax[r,1].set_title('Defect mask')
            ax[r,2].imshow(images_np[idx]); ax[r,2].imshow(score_maps['shallow'][idx], cmap='inferno', alpha=0.58, vmin=vmin, vmax=vmax)
            ax[r,2].set_title('Shallow flow')
            ax[r,3].imshow(images_np[idx]); ax[r,3].imshow(fused[idx], cmap='inferno', alpha=0.58, vmin=vmin, vmax=vmax)
            ax[r,3].set_title('Fused flow')
            for col in range(4): ax[r,col].axis('off')
            selection_rows.append({'path':paths_np[idx],
                'shallow image AUPRC':float(average_precision_score(masks_np[idx].reshape(-1),score_maps['shallow'][idx].reshape(-1))),
                'fused image AUPRC':float(average_precision_score(masks_np[idx].reshape(-1),fused[idx].reshape(-1)))})
        savefig(f'part1_{category}_examples.png')
        pd.DataFrame(selection_rows).to_csv(ROOT/f'mvtec_{category}_display_selection.csv', index=False)
    print('Evaluated all test images; displayed subset is for illustration only.')

mvtec_df = pd.DataFrame(mvtec_rows)
mvtec_df.to_csv(ROOT/'mvtec_results.csv', index=False)
display(mvtec_df)
print('MVTec results saved to:', ROOT/'mvtec_results.csv')


**Question I.7 (11 points).** Use the table and the visual examples to compare the MVTec categories.

1. Which category achieves the best **fused pixel AUROC** in your run?
2. In your figures, when is the fused map visually better than the shallow map?
3. Give one limitation of likelihood-based anomaly detection that is visible in these examples.

**Answer:** _Write your answer here._


# **Part II — Conditional Colorization (30 points)**

In this part we model the conditional distribution of image chrominance given grayscale luminance.
We convert RGB images to **YUV** and keep luminance $Y$ at full resolution, while the chrominance $(U,V)$ is modeled at a lower spatial resolution.

The model has two pieces:
1. a grayscale conditioning network that extracts features from the luminance image,
2. a conditional flow that models the chrominance distribution.

This experiment is different from deterministic regression. One grayscale image can correspond to several plausible colors, so we sample multiple outputs from the conditional flow.


In [ ]:
# STL-10 hosted as Parquet on Hugging Face (the original torchvision host often refuses connections).
# Only the labeled train and test splits are loaded; the 100k unlabeled images are NOT downloaded.
# Cached files are reused by Hugging Face Datasets across later notebook runs.

def load_stl10_datasets():
    try:
        from datasets import load_dataset
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'datasets'])
        from datasets import load_dataset

    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
    ])
    eval_tf = transforms.Compose([
        transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
        transforms.ToTensor(),
    ])

    # Two public mirrors with matching STL-10 labeled splits.
    # Keep the same repository for both splits.
    repos = ('mteb/stl10', 'tanganke/stl10')
    last_error = None
    for repo in repos:
        try:
            print('Loading STL-10 from', repo)
            full_train = load_dataset(repo, split='train')
            full_test = load_dataset(repo, split='test')
            assert len(full_train) == 5000 and len(full_test) == 8000, (
                f'Unexpected split sizes for {repo}: {len(full_train)}, {len(full_test)}'
            )
            print('Loaded', len(full_train), 'training and', len(full_test), 'test images')
            break
        except Exception as exc:
            last_error = exc
            print(f'Mirror {repo} unavailable ({type(exc).__name__}): {str(exc)[:180]}')
    else:
        raise RuntimeError(
            'STL-10 could not be downloaded from either Hugging Face mirror. '
            'Enable Kaggle Internet or check whether the dataset is cached.'
        ) from last_error

    class STL10Images(Dataset):
        def __init__(self, hf_split, transform):
            self.data = hf_split
            self.transform = transform
        def __len__(self):
            return len(self.data)
        def __getitem__(self, index):
            image = self.data[int(index)]['image']
            if not isinstance(image, Image.Image):
                raise TypeError(f'Expected a PIL image, received {type(image)}')
            return self.transform(image.convert('RGB'))

    # Validation is a fixed 500-image subset of the official TRAIN split.
    # Unlike training, validation does not use random horizontal flips.
    val_count = 500
    perm = torch.randperm(len(full_train), generator=torch.Generator().manual_seed(SEED)).tolist()
    val_set = STL10Images(full_train.select(perm[:val_count]), eval_tf)
    train_set = STL10Images(full_train.select(perm[val_count:]), train_tf)
    test_set = STL10Images(full_test, eval_tf)  # Official held-out TEST split; no training on it.
    return train_set, val_set, test_set

train_set, valid_set, test_set = load_stl10_datasets()
# Zero workers avoids Hugging Face Dataset serialization issues on Windows/macOS notebooks.
train_loader = DataLoader(train_set, batch_size=STL_BATCH_SIZE, shuffle=True, num_workers=0,
                          pin_memory=(DEVICE.type == 'cuda'), drop_last=True)
valid_loader = DataLoader(valid_set, batch_size=16, shuffle=False, num_workers=0,
                          pin_memory=(DEVICE.type == 'cuda'))
print('STL-10 sizes | train:', len(train_set), '| valid:', len(valid_set), '| test:', len(test_set))


In [ ]:

def rgb_to_yuv(x):
    """
    Input:  x in [0,1] with shape [B,3,H,W]
    Output: Y with shape [B,1,H,W], UV with shape [B,2,H,W]
    """
    # TODO II.1 (8 points): implement the RGB -> YUV conversion.
    raise NotImplementedError('TODO II.1 rgb_to_yuv')

def yuv_to_rgb(y, uv, clamp=True):
    # TODO II.1 continuation (4 of the 8 points): implement YUV -> RGB.
    raise NotImplementedError('TODO II.1 yuv_to_rgb')

def split_image(x):
    y, uv = rgb_to_yuv(x)
    uv = F.interpolate(uv, size=(CHROMA_SIZE, CHROMA_SIZE), mode='area')
    return y, uv

def restore_image(y_full, uv_small):
    uv_full = F.interpolate(uv_small, size=y_full.shape[-2:], mode='bilinear', align_corners=False)
    return yuv_to_rgb(y_full, uv_full)

def visualization_rgb(t):
    return t.detach().cpu().permute(1, 2, 0).numpy().clip(0, 1)

def training_batches(loader):
    while True:
        for item in loader:
            yield item

train_vis = next(iter(train_loader))
y_vis, uv_vis = split_image(train_vis)
fig, axes = plt.subplots(2, 6, figsize=(15, 5))
for i, ax in enumerate(axes[0]):
    ax.imshow(visualization_rgb(train_vis[i])); ax.axis('off')
for i, ax in enumerate(axes[1]):
    ax.imshow(y_vis[i, 0].numpy(), cmap='gray', vmin=0, vmax=1); ax.axis('off')
axes[0, 0].set_title('STL-10 RGB'); axes[1, 0].set_title('Luminance')
savefig('part3_stl_preview.png')


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, channels):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1), nn.SiLU(),
            nn.Conv2d(channels, channels, 3, padding=1),
        )
    def forward(self, x):
        return F.silu(x + 0.5*self.net(x))

class GrayUNet(nn.Module):
    def __init__(self, width=48, out_channels=48):
        super().__init__()
        self.in_conv = nn.Conv2d(1, width, 3, padding=1)
        self.l0 = ResidualBlock(width)
        self.d1 = nn.Conv2d(width, width*2, 3, stride=2, padding=1)
        self.l1 = ResidualBlock(width*2)
        self.d2 = nn.Conv2d(width*2, width*2, 3, stride=2, padding=1)
        self.mid = nn.Sequential(ResidualBlock(width*2), ResidualBlock(width*2))
        self.up1 = nn.Conv2d(width*4, width*2, 3, padding=1)
        self.r1 = ResidualBlock(width*2)
        self.up0 = nn.Conv2d(width*3, width, 3, padding=1)
        self.r0 = ResidualBlock(width)
        self.out = nn.Conv2d(width, out_channels, 3, padding=1)
    def forward(self, y_small):
        s0 = self.l0(self.in_conv(y_small))
        s1 = self.l1(self.d1(s0))
        m = self.mid(self.d2(s1))
        m = F.interpolate(m, size=s1.shape[-2:], mode='nearest')
        m = self.r1(self.up1(torch.cat((m, s1), dim=1)))
        m = F.interpolate(m, size=s0.shape[-2:], mode='nearest')
        m = self.r0(self.up0(torch.cat((m, s0), dim=1)))
        return self.out(m)

class ActNorm(nn.Module):
    def __init__(self, channels=2):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.log_scale = nn.Parameter(torch.zeros(1, channels, 1, 1))
        self.register_buffer('initialized', torch.tensor(False))
    @torch.no_grad()
    def initialize(self, x):
        mu = x.mean(dim=(0,2,3), keepdim=True)
        sigma = x.std(dim=(0,2,3), keepdim=True).clamp_min(1e-5)
        self.bias.copy_(-mu)
        self.log_scale.copy_(-sigma.log())
        self.initialized.fill_(True)
    def forward(self, x, reverse=False):
        if not bool(self.initialized):
            if reverse:
                raise RuntimeError('ActNorm has not been initialized.')
            self.initialize(x)
        if reverse:
            return x * torch.exp(-self.log_scale) - self.bias
        b, _, h, w = x.shape
        z = (x + self.bias) * torch.exp(self.log_scale)
        ld = (h*w*self.log_scale.sum()).expand(b)
        return z, ld

class Mix1x1(nn.Module):
    def __init__(self, channels=2):
        super().__init__()
        w = torch.empty(channels, channels)
        nn.init.orthogonal_(w)
        self.weight = nn.Parameter(w)
    def forward(self, x, reverse=False):
        b, c, h, w = x.shape
        if reverse:
            inv = torch.linalg.inv(self.weight)
            return F.conv2d(x, inv.reshape(c, c, 1, 1))
        z = F.conv2d(x, self.weight.reshape(c, c, 1, 1))
        sign, logabsdet = torch.linalg.slogdet(self.weight)
        return z, (h*w*logabsdet).expand(b)

class CouplingNet(nn.Module):
    def __init__(self, conditioner_channels, hidden=64):
        super().__init__()
        self.c1 = nn.Conv2d(2 + 1 + conditioner_channels, hidden, 3, padding=1)
        self.r = nn.Sequential(ResidualBlock(hidden), ResidualBlock(hidden))
        self.cout = nn.Conv2d(hidden, 4, 3, padding=1)
        nn.init.zeros_(self.cout.weight)
        nn.init.zeros_(self.cout.bias)
    def forward(self, masked_chroma, gray, condition):
        x = torch.cat((masked_chroma, gray, condition), dim=1)
        return self.cout(self.r(F.silu(self.c1(x)))).chunk(2, dim=1)

class CheckerboardCoupling(nn.Module):
    def __init__(self, cond_channels, hidden, parity):
        super().__init__()
        self.parity = parity
        self.net = CouplingNet(cond_channels, hidden)
    def forward(self, x, gray, cond, reverse=False):
        # TODO II.2 (10 points): implement the checkerboard affine coupling.
        # Requirements:
        # 1. Build a binary checkerboard mask with the given parity.
        # 2. Keep the masked part unchanged.
        # 3. Predict shift and logscale from the masked chroma, grayscale image, and condition tensor.
        # 4. Apply the affine transform only on the unmasked spatial locations.
        # 5. Bound the logscale with 0.8 * tanh(logscale).
        # 6. In reverse mode, invert the affine transform and return only the reconstructed tensor.
        # 7. In forward mode, also return the summed log-determinant for each sample.
        raise NotImplementedError('TODO II.2')

class FlowStep(nn.Module):
    def __init__(self, cond_channels, hidden, parity):
        super().__init__()
        self.norm = ActNorm(2)
        self.mix = Mix1x1(2)
        self.couple = CheckerboardCoupling(cond_channels, hidden, parity)
    def forward(self, x, gray, cond, reverse=False):
        if reverse:
            x = self.couple(x, gray, cond, reverse=True)
            x = self.mix(x, reverse=True)
            return self.norm(x, reverse=True)
        x, d1 = self.norm(x)
        x, d2 = self.mix(x)
        x, d3 = self.couple(x, gray, cond, reverse=False)
        return x, d1 + d2 + d3

class ConditionalChromaFlow(nn.Module):
    def __init__(self, steps=FLOW_STEPS, conditioner_width=COND_WIDTH, hidden=FLOW_HIDDEN):
        super().__init__()
        self.conditioner = GrayUNet(width=conditioner_width, out_channels=conditioner_width)
        self.steps = nn.ModuleList([FlowStep(conditioner_width, hidden, i % 2) for i in range(steps)])
    def gray_condition(self, y_full):
        small = F.interpolate(y_full, size=(CHROMA_SIZE, CHROMA_SIZE), mode='area')
        return small, self.conditioner(small)
    def encode(self, uv, gray_full):
        gray, cond = self.gray_condition(gray_full)
        z = uv
        total_ld = torch.zeros(uv.shape[0], device=uv.device)
        for f in self.steps:
            z, ld = f(z, gray, cond, reverse=False)
            total_ld = total_ld + ld
        return z, total_ld
    def log_prob(self, uv, gray_full):
        z, ld = self.encode(uv, gray_full)
        prior_lp = -0.5 * (z.square() + math.log(2*math.pi)).flatten(1).sum(1)
        return prior_lp + ld
    def sample(self, gray_full, samples=4, temperature=0.9):
        gray, cond = self.gray_condition(gray_full)
        b, _, h, w = gray.shape
        zz = torch.randn(b*samples, 2, h, w, device=gray.device, dtype=gray.dtype) * temperature
        gr = gray.repeat_interleave(samples, dim=0)
        cc = cond.repeat_interleave(samples, dim=0)
        for f in reversed(self.steps):
            zz = f(zz, gr, cc, reverse=True)
        return zz.view(b, samples, 2, h, w)

model = ConditionalChromaFlow().to(DEVICE)
dummy_rgb = next(iter(train_loader))[:2].to(DEVICE)
dummy_y, dummy_uv = split_image(dummy_rgb)
z, ld = model.encode(dummy_uv, dummy_y)
y_small, cond = model.gray_condition(dummy_y)
recovered = z
for block in reversed(model.steps):
    recovered = block(recovered, y_small, cond, reverse=True)
inv_error = (dummy_uv - recovered).abs().max().item()
assert inv_error < 1e-3, f'invertibility failure: {inv_error:.2e}'
loss_check = -model.log_prob(dummy_uv, dummy_y).mean() / dummy_uv[0].numel()
loss_check.backward()
assert all(torch.isfinite(p.grad).all() for p in model.parameters() if p.grad is not None)
model.zero_grad(set_to_none=True)
print('Part II invertibility and gradient checks passed. Parameters:', f'{sum(p.numel() for p in model.parameters())/1e6:.2f}M')


In [ ]:
def conditional_nll_per_dim(model, chroma, gray):
    """Return scalar mean negative conditional log-likelihood per chroma dimension.

    chroma [B,2,h,w]; gray [B,1,H,W]. Preserve differentiability.
    """
    # TODO II.3 (4 points): implement mean NLL divided by chroma dimensions per sample.
    raise NotImplementedError('TODO II.3')

# Small, cheap correctness check before expensive optimization.
_test_y, _test_uv = split_image(dummy_rgb)
_test_nll = conditional_nll_per_dim(model, _test_uv, _test_y)
assert _test_nll.ndim == 0 and torch.isfinite(_test_nll)
assert torch.allclose(_test_nll, -model.log_prob(_test_uv, _test_y).mean() / _test_uv[0].numel(), rtol=1e-5, atol=1e-5)
print('Conditional NLL per dimension check passed.')


optimizer = torch.optim.Adam(model.parameters(), lr=LR)
checkpoint_path = CKPT_ROOT / 'stl10_conditional_flow_latest.pt'
history, val_history = [], []
stream = training_batches(train_loader)
start = time.time()
for step in range(1, COLORIZATION_TRAIN_STEPS + 1):
    model.train()
    rgb = next(stream).to(DEVICE, non_blocking=True)
    gray, uv = split_image(rgb)
    nll = conditional_nll_per_dim(model, uv, gray)
    if not torch.isfinite(nll):
        raise RuntimeError(f'Non-finite colorization loss at step {step}')
    optimizer.zero_grad(set_to_none=True)
    nll.backward()
    nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
    optimizer.step()
    history.append([step, float(nll.detach().cpu())])
    if step == 1 or step % LOG_EVERY == 0:
        print(f'step {step}/{COLORIZATION_TRAIN_STEPS} | train NLL/dim {history[-1][1]:.4f}')
    if step % VALIDATE_EVERY == 0 or step == COLORIZATION_TRAIN_STEPS:
        model.eval()
        vals = []
        with torch.no_grad():
            for vi, vrgb in enumerate(valid_loader):
                if vi >= 4:
                    break
                vrgb = vrgb.to(DEVICE, non_blocking=True)
                vy, vc = split_image(vrgb)
                vals.append(float(conditional_nll_per_dim(model, vc, vy).cpu()))
        val_history.append([step, float(np.mean(vals))])
        print(f'       held-out NLL/dim {val_history[-1][1]:.4f}')
    if step % SAVE_EVERY == 0 or step == COLORIZATION_TRAIN_STEPS:
        torch.save({
            'step': step, 'model': model.state_dict(), 'optimizer': optimizer.state_dict(),
            'history': history, 'val_history': val_history
        }, checkpoint_path)
print('Training duration (minutes):', round((time.time() - start) / 60, 2))

fig, ax = plt.subplots(figsize=(8.8, 4.3))
if history:
    hh = np.asarray(history)
    ax.plot(hh[:,0], hh[:,1], label='train NLL/dim', linewidth=1.1)
if val_history:
    vv = np.asarray(val_history)
    ax.plot(vv[:,0], vv[:,1], '-o', label='held-out NLL/dim', markersize=4)
ax.set_xlabel('Training step')
ax.set_ylabel('Negative log-likelihood / chroma dimension')
ax.grid(alpha=0.25)
ax.legend()
savefig('part3_nll_curve.png')

model.eval()
valid_rgb = torch.stack([valid_set[i] for i in range(NUM_VIS_IMAGES)], dim=0).to(DEVICE)
gray, uv_truth = split_image(valid_rgb)
with torch.no_grad():
    uv_samples = model.sample(gray, samples=NUM_SAMPLES, temperature=0.9)

cols = 3 + NUM_SAMPLES
fig, axes = plt.subplots(NUM_VIS_IMAGES, cols, figsize=(2.8*cols, 2.7*NUM_VIS_IMAGES), squeeze=False)
column_titles = ['Grayscale', 'Ground truth', 'Oracle half-res chroma'] + [f'Flow sample {j+1}' for j in range(NUM_SAMPLES)]
for j, label in enumerate(column_titles):
    axes[0,j].set_title(label, fontsize=9)
for i in range(NUM_VIS_IMAGES):
    y_i = gray[i:i+1]
    images = [y_i.repeat(1,3,1,1)[0], valid_rgb[i], restore_image(y_i, uv_truth[i:i+1])[0]]
    images += [restore_image(y_i, uv_samples[i,j:j+1])[0] for j in range(NUM_SAMPLES)]
    for j, im in enumerate(images):
        axes[i,j].imshow(visualization_rgb(im)); axes[i,j].axis('off')
savefig('part3_colorization_samples.png', dpi=150)

show_images = min(3, NUM_VIS_IMAGES)
fig, ax = plt.subplots(show_images, 1 + len(TEMPERATURES)*2, figsize=(2.85*(1+len(TEMPERATURES)*2), 2.9*show_images), squeeze=False)
for i in range(show_images):
    y_i = gray[i:i+1]
    ax[i,0].imshow(visualization_rgb(y_i.repeat(1,3,1,1)[0])); ax[i,0].axis('off')
    if i == 0:
        ax[i,0].set_title('Grayscale')
    for j, T in enumerate(TEMPERATURES):
        with torch.no_grad():
            sample_uv = model.sample(y_i, samples=2, temperature=T)[0]
        for k in range(2):
            col = 1 + 2*j + k
            rgb = restore_image(y_i, sample_uv[k:k+1])[0]
            ax[i,col].imshow(visualization_rgb(rgb)); ax[i,col].axis('off')
            if i == 0:
                ax[i,col].set_title(f'T={T}, #{k+1}')
savefig('part3_temperature_comparison.png', dpi=160)

with torch.no_grad():
    sampled_rgb = torch.stack([restore_image(gray, uv_samples[:,s]) for s in range(NUM_SAMPLES)], dim=1)
    y_reconstructed = rgb_to_yuv(sampled_rgb.flatten(0,1))[0].reshape(NUM_VIS_IMAGES, NUM_SAMPLES, 1, IMAGE_SIZE, IMAGE_SIZE)
    luminance_mae = (y_reconstructed - gray[:,None]).abs().mean().item()
    chroma_diversity = (uv_samples[:,1:] - uv_samples[:,:1]).abs().mean().item()
    true_magnitude = uv_truth.abs().mean().item()
    sample_magnitude = uv_samples.abs().mean().item()
summary = {
    'luminance_mae_after_rgb_clipping': luminance_mae,
    'mean_pairwise_chroma_difference_from_first': chroma_diversity,
    'true_chroma_abs_mean': true_magnitude,
    'sampled_chroma_abs_mean': sample_magnitude,
    'training_steps': COLORIZATION_TRAIN_STEPS,
}
print(json.dumps(summary, indent=2))
(Path(ROOT) / 'part3_diagnostics.json').write_text(json.dumps(summary, indent=2))


**Question II.4 (8 points).** Use your figures and diagnostics to answer the following.

1. Why is held-out NLL not the same as visual quality in this experiment?
2. What happens when the sampling temperature increases from `0.65` to `1.1`?
3. Why is the luminance preserved only approximately after RGB reconstruction and clipping?

**Answer:** _Write your answer here._


# **Part III — Color Transfer (10 points)**

The final experiment is intentionally simpler than Part II.
We no longer predict pixelwise colors from grayscale input. Instead, we learn a **global** density on the 2D chroma vector $(a^*, b^*)$ in CIE Lab space.

We fit one masked autoregressive flow (MAF) to the source palette and another to the target palette.
A source image is encoded into the latent space of the source flow and decoded through the target flow.
This changes the global chroma distribution while approximately preserving lightness.

This part is easier than conditional colorization because it ignores spatial semantics.
A global pixel-chroma model can transfer a palette, but it does not know which object should receive which color.


In [ ]:

def photo_float(image, side=320):
    im = np.asarray(image)
    im = skresize(im, (side, side, 3), anti_aliasing=True, preserve_range=True)
    return np.clip(im / 255.0, 0, 1).astype('float32')

base_photo = photo_float(skdata.rocket())
reference_photo = photo_float(skdata.coffee())

def grade_photo(image, mood):
    lab = skcolor.rgb2lab(image).astype('float32')
    a, b = lab[:,:,1], lab[:,:,2]
    if mood == 'cool':
        lab[:,:,1] = 0.78*a - 6
        lab[:,:,2] = 0.78*b - 19
    elif mood == 'warm':
        lab[:,:,1] = 0.78*a + 13
        lab[:,:,2] = 0.78*b + 24
    else:
        raise ValueError(mood)
    return np.clip(skcolor.lab2rgb(lab), 0, 1).astype('float32')

source_photo = grade_photo(base_photo, 'cool')
warm_reference = grade_photo(base_photo, 'warm')

class MaskedLinear2D(nn.Linear):
    def __init__(self, ins, outs, mask):
        super().__init__(ins, outs)
        self.register_buffer('mask', mask.float())
    def forward(self, x):
        return F.linear(x, self.weight * self.mask, self.bias)

class MADE2D(nn.Module):
    def __init__(self, width=64):
        super().__init__()
        d_in = torch.arange(1, 3)
        hdeg = torch.ones(width, dtype=torch.int64)
        d_out = torch.arange(1, 3).repeat(2)
        self.net = nn.Sequential(
            MaskedLinear2D(2, width, hdeg[:,None] >= d_in[None,:]), nn.Tanh(),
            MaskedLinear2D(width, width, hdeg[:,None] >= hdeg[None,:]), nn.Tanh(),
            MaskedLinear2D(width, 4, d_out[:,None] > hdeg[None,:]),
        )
    def forward(self, x):
        mu, log_s = self.net(x).chunk(2, dim=-1)
        return mu, 1.5*torch.tanh(log_s / 1.5)

class ChromaMAF(nn.Module):
    def __init__(self, layers=6, width=64):
        super().__init__()
        self.mades = nn.ModuleList([MADE2D(width) for _ in range(layers)])
        self.register_buffer('perm', torch.tensor([1, 0]))
    def encode(self, x):
        # TODO III.1 (6 points): implement the forward pass of the MAF.
        # Return latent z and the accumulated log-determinant.
        raise NotImplementedError('TODO III.1 encode')
    def decode(self, z):
        for net in reversed(self.mades):
            x = torch.zeros_like(z)
            for i in range(2):
                mu, s = net(x)
                x[:, i] = z[:, i] * torch.exp(s[:, i]) + mu[:, i]
            z = x[:, self.perm]
        return z
    def log_prob(self, x):
        # TODO III.1 continuation (2 of the 6 points): use encode() to compute log p(x).
        raise NotImplementedError('TODO III.1 log_prob')

_maf = ChromaMAF(layers=3, width=24).to(DEVICE)
_x = torch.randn(32, 2, device=DEVICE) * 0.5
_z, _ = _maf.encode(_x)
_inv = _maf.decode(_z)
err = (_x - _inv).abs().max().item()
assert err < 1e-5, f'MAF inverse check failed: {err:.2e}'
(-_maf.log_prob(_x).mean()).backward()
assert any(p.grad is not None and p.grad.abs().sum() > 0 for p in _maf.parameters())
print('Part III inverse and gradient checks passed.')


In [ ]:

def chroma_pixels(image):
    lab = skcolor.rgb2lab(np.clip(image, 0, 1))
    return np.asarray(lab[:,:,1:3] / 64.0, dtype='float32').reshape(-1, 2)

def fit_chroma_maf(image, seed_offset, steps=MAF_STEPS):
    torch.manual_seed(SEED + seed_offset)
    model = ChromaMAF(layers=6, width=64).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=6e-4)
    samples = torch.from_numpy(chroma_pixels(image)).to(DEVICE)
    history = []
    for step in range(1, steps + 1):
        ids = torch.randint(len(samples), (1024,), device=DEVICE)
        x = samples[ids] + 0.002 * torch.randn(1024, 2, device=DEVICE)
        loss = -model.log_prob(x).mean()
        if not torch.isfinite(loss):
            raise RuntimeError(f'Chroma MAF diverged at step {step}')
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 5.0)
        optimizer.step()
        history.append(float(loss.detach().cpu()))
    return model.eval(), history

_D = 6/29
_RGB_M = np.array([[ 3.2404542,-1.5371385,-0.4985314],
                   [-0.9692660, 1.8760108, 0.0415560],
                   [ 0.0556434,-0.2040259, 1.0572252]], dtype=np.float64)

def lab_to_linear_rgb(lab):
    lab = np.asarray(lab, dtype=np.float64)
    L, a, b = lab[...,0], lab[...,1], lab[...,2]
    fy = (L + 16.0) / 116.0
    fx = fy + a / 500.0
    fz = fy - b / 200.0
    def invf(t):
        return np.where(t > _D, t**3, 3*_D**2*(t - 4/29))
    xyz = np.stack([0.95047*invf(fx), invf(fy), 1.08883*invf(fz)], axis=-1)
    return xyz @ _RGB_M.T

def linear_to_srgb(rgb):
    rgb = np.clip(rgb, 0, 1)
    return np.where(rgb <= 0.0031308, 12.92*rgb, 1.055*np.power(rgb, 1/2.4)-0.055)

def gamut_safe_lab_to_rgb(lab, iterations=16):
    lab = np.asarray(lab, dtype=np.float64)
    lo = np.zeros(lab.shape[:2], dtype=np.float64)
    hi = np.ones_like(lo)
    for _ in range(iterations):
        mid = (lo + hi) / 2
        candidate = lab.copy()
        candidate[...,1:3] = lab[...,1:3] * mid[...,None]
        rgb = lab_to_linear_rgb(candidate)
        valid = np.isfinite(rgb).all(-1) & (rgb >= 0).all(-1) & (rgb <= 1).all(-1)
        lo = np.where(valid, mid, lo)
        hi = np.where(valid, hi, mid)
    candidate = lab.copy()
    candidate[...,1:3] *= lo[...,None]
    rgb = linear_to_srgb(lab_to_linear_rgb(candidate))
    return np.clip(rgb, 0, 1).astype(np.float32), float((lo < 0.999).mean())

def transfer_lab_maf(image, source_flow, target_flow, strength=0.80, chunk=8192):
    lab = skcolor.rgb2lab(image).astype('float64')
    original_lab = lab.copy()
    chroma = lab[...,1:3].reshape(-1, 2) / 64.0
    src = torch.from_numpy(chroma.astype(np.float32)).to(DEVICE)
    transported = []
    with torch.no_grad():
        for batch in src.split(chunk):
            z, _ = source_flow.encode(batch)
            transported.append(target_flow.decode(z).cpu().numpy())
    new_chroma = np.concatenate(transported).reshape(*lab.shape[:2], 2) * 64.0
    lab[...,1:3] = (1-strength) * original_lab[...,1:3] + strength * new_chroma
    result, compressed_fraction = gamut_safe_lab_to_rgb(lab)
    actual_L = skcolor.rgb2lab(result)[...,0]
    lightness_change = float(np.abs(actual_L - original_lab[...,0]).mean())
    return result, compressed_fraction, lightness_change

cold_flow, cold_hist = fit_chroma_maf(source_photo, seed_offset=1)
warm_flow, warm_hist = fit_chroma_maf(warm_reference, seed_offset=2)
ref_flow, ref_hist = fit_chroma_maf(reference_photo, seed_offset=3)

# Controlled transfer: same scene, cool -> warm.
test_lab = skcolor.rgb2lab(source_photo)
test_rgb, _ = gamut_safe_lab_to_rgb(test_lab)
maxerr = float(np.max(np.abs(test_rgb - source_photo)))
print('RGB/Lab valid-color round-trip max error:', round(maxerr, 5))
controlled_result, compressed, mean_l_change = transfer_lab_maf(source_photo, cold_flow, warm_flow, strength=0.80)
fig, axes = plt.subplots(1, 3, figsize=(11.8, 4))
for ax, im, title in zip(axes, [source_photo, controlled_result, warm_reference],
                         ['Cool source', 'MAF result (strength 0.80)', 'Warm reference']):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
savefig('part4_controlled_transfer.png', dpi=165)
print(f'Out-of-gamut chroma compressed in {100*compressed:.1f}% of pixels; mean |ΔL*|={mean_l_change:.3f}.')

# Cross-image palette transfer.
palette_result, compressed2, mean_l_change2 = transfer_lab_maf(source_photo, cold_flow, ref_flow, strength=0.75)
fig, axes = plt.subplots(1, 3, figsize=(11.8, 4))
for ax, im, title in zip(axes, [source_photo, palette_result, reference_photo],
                         ['Source image', 'Transferred palette', 'Palette reference']):
    ax.imshow(im); ax.set_title(title); ax.axis('off')
savefig('part4_cross_palette_transfer.png', dpi=165)
print(f'Cross-photo transfer: {100*compressed2:.1f}% gamut compression, mean |ΔL*|={mean_l_change2:.3f}.')


**Question III.2 (4 points).** Compare Part III with Part II.
Why is this global palette-transfer problem easier than conditional colorization, and why can the MAF still produce semantically wrong colors even when the overall palette looks reasonable?

**Answer:** _Write your answer here._


## References

1. Dinh, L., Sohl-Dickstein, J., and Bengio, S. **Density Estimation using Real NVP.** ICLR 2017.
2. Papamakarios, G., Pavlakou, T., and Murray, I. **Masked Autoregressive Flow for Density Estimation.** NeurIPS 2017.
3. Bergmann, P., Fauser, M., Sattlegger, D., and Steger, C. **MVTec AD — A Comprehensive Real-World Dataset for Unsupervised Anomaly Detection.** CVPR 2019.
4. Coates, A., Ng, A., and Lee, H. **An Analysis of Single-Layer Networks in Unsupervised Feature Learning.** AISTATS 2011. (STL-10 dataset)



